In [1]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../env/.env")

True

In [17]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chains import PebbloRetrievalQA
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
import gradio as gr

In [19]:
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview",temperature=0)

In [4]:
#Prompt
template = """
You are an HR Policy expoert at Nestle company. 
Answer the question using ONLY the context below. 
If answer is not found in the context, say 'I don't know.'
Context:{context}

Answer:
"""

In [5]:
prompt = ChatPromptTemplate.from_messages([
        ("system", template),
        ("human","{question}")
    ]
)

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [8]:
vector_store = Chroma(persist_directory="../chroma_db",embedding_function=embeddings)

/tmp/ipykernel_362405/3895024317.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory="../chroma_db",embedding_function=embeddings)


In [9]:
retreiver = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [20]:
rag_chain = (
    {
        "context" : retreiver | format_docs,
        "question" : RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
test = rag_chain.invoke("Leadership")
print(test)

Based on the Nestlé Human Resources Policy:

*   **Responsibility:** Line managers have the prime responsibility for building and sustaining a committed work environment and for caring for and developing the leaders of tomorrow.
*   **Decision Making:** Line managers act as the final decision makers on all people matters under their influence, within the boundaries of company policies and principles.
*   **Development:** Corporate leadership programmes are used to develop and retain the best-qualified management. Leaders have the opportunity to attend international training courses. 
*   **Training Philosophy:** Attending a leadership programme is considered a component of ongoing development rather than a reward.
*   **Functional Leadership:** HR ensures functional leadership through a streamlined structure consisting of Centres of Expertise, Business Partners, and Employee Services.


In [26]:
    # embed message, find similar chunk from chroma db, retreive top 3 chunks
    # Create prompt template for system promt - assign role, provide history, chunks, current query
    # receive response
    # return response
def chat_fn(question, history):
    return rag_chain.invoke(question)

In [34]:
demo  = gr.ChatInterface(fn=chat_fn, title="Nestle HR Policy Assistant", description="Ask questions about Nestle's HR policies",
                         examples=[
                             "Employee Relations",
                             "Total Rewards Program",
                             "Talent and Development",
                             "Training and Learning",
                             "Onboarding",
                             "Leadership and Management"
                         ])

In [35]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://34388d00d3b37b2678.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#demo.close()

Closing server running on port: 7862
